# Multi-Agent Evidence-Graded Specialist RAG

Runs Architecture 1: Evidence-Graded Specialist Pipeline — a 5-agent sequential pipeline
that routes each question to the appropriate retrieval strategy, selects the top-k context
chunks, generates an answer with evidence-based reasoning, and flags unsupported claims
via a detection-only hallucination guard.

**Pipeline:** Question → Agent 1 (Router) → Agent 2 (Adaptive Retrieval) → Agent 3 (Evidence Selection) → Agent 4 (Answer Generator) → Agent 5 (Hallucination Guard) → Answer + Flags  
**Evaluation:** RAGAS and DeepEval metrics

In [5]:
import sys
sys.path.append("..")

import os
import re
import time
import json
import pandas as pd
from datetime import datetime
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
import config
from ast import literal_eval
from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
from ragas import SingleTurnSample
from deepeval.evaluate import DisplayConfig, AsyncConfig
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import (
    FaithfulnessMetric, ContextualRecallMetric,
    ContextualPrecisionMetric, AnswerRelevancyMetric, GEval
)
import deepeval
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()
pd.set_option('display.html.use_mathjax', False)
os.environ["DEEPEVAL_RETRY_MAX_ATTEMPTS"] = "2"

import logging
logging.basicConfig(level=logging.ERROR)

/tmp/ipykernel_8500/4217960929.py:18: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams
/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_8500/4217960929.py:21: DeprecationWarning: Importing NonLLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMContextRecall
  from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, 

## Load Vector Store and Retriever Helpers

Reuses the same retriever construction functions as all prior experiments.

In [6]:
def load_chroma(embeddings, db_name=None, persist_dir=None):
    """Load an existing ChromaDB collection from disk."""
    from langchain_chroma import Chroma

    db_name = db_name or f"{config.DEFAULT_EMBEDDING}_pubmed_chromadb"
    persist_dir = persist_dir or str(config.VECTORSTORE_DIR / db_name)
    print(f"Loading ChromaDB from {persist_dir}")
    return Chroma(
        collection_name=db_name,
        persist_directory=persist_dir,
        embedding_function=embeddings,
    )


def get_cosine_retriever(vector_store, k=None):
    """Build a cosine similarity retriever."""
    k = k or config.TOP_K
    return vector_store.as_retriever(search_kwargs={"k": k})


def get_bm25_retriever(vector_store, k=None):
    """Build a BM25 sparse retriever from the full stored corpus."""
    k = k or config.TOP_K
    docs = [
        Document(page_content=x, metadata=m)
        for x, m in zip(vector_store.get()["documents"], vector_store.get()["metadatas"])
    ]
    retriever = BM25Retriever.from_documents(docs)
    retriever.k = k
    return retriever


def rrf(rank_lists, top_k=None, k=60):
    """Fuse multiple ranked retrieval lists using Reciprocal Rank Fusion."""
    top_k = top_k or config.TOP_K
    scores = defaultdict(lambda: {"doc": None, "score": 0.0})
    for docs in rank_lists:
        for rank, doc in enumerate(docs, 1):
            key = (doc.metadata["pubid"], doc.metadata["chunk_index"])
            if scores[key]["doc"] is None:
                scores[key]["doc"] = doc
            scores[key]["score"] += 1 / (k + rank)
    return [x["doc"] for x in sorted(scores.values(), key=lambda x: x["score"], reverse=True)[:top_k]]


def invoke_hybrid_retriever(query, dense_retriever, sparse_retriever, top_k=None):
    """Hybrid RRF retrieval: dense + BM25, fused by Reciprocal Rank Fusion."""
    top_k = top_k or config.TOP_K
    dense_docs = dense_retriever.invoke(query)
    sparse_docs = sparse_retriever.invoke(query)
    return rrf([dense_docs, sparse_docs], top_k=top_k)

In [7]:
def get_groq_llm(model=None, api_key=None):
    return ChatGroq(
        model=model or config.LLM_MODEL,
        api_key=api_key or config.GROQ_API_KEY,
    )


class GroqKeyRotator:
    """Cycles through config.GROQ_API_KEYS, rebuilding the LLM client on each rotation."""

    def __init__(self, model=None):
        if not config.GROQ_API_KEYS:
            raise ValueError("config.GROQ_API_KEYS is empty — set GROQ_API_KEY or GROQ_API_KEYS in .env")
        self.api_keys = config.GROQ_API_KEYS
        self.model = model or config.LLM_MODEL
        self.current_idx = 0
        print(f"Initialized GroqKeyRotator with {len(self.api_keys)} API key(s)")

    def get_llm(self):
        """Return a ChatGroq instance using the current API key."""
        return ChatGroq(
            model=self.model,
            api_key=self.api_keys[self.current_idx],
        )

    def rotate(self):
        """Advance to the next key, wrapping around."""
        self.current_idx = (self.current_idx + 1) % len(self.api_keys)
        print(f"Rotated to API key index {self.current_idx}")


RAG_PROMPT_TEMPLATE = """You are a biomedical research assistant. Use the following research contexts to answer the question.

Context:
{context}

Question: {question}

Instructions:
- Synthesize an answer from the provided context using evidence-based reasoning.
- Draw logical inferences and conclusions from the evidence when a direct statement is not available.
- Cite specific findings, statistics, or conclusions from the context to support your answer.
- Do NOT refuse to answer or state that the context is insufficient. Use whatever relevant evidence is available.

Answer:"""

RAG_PROMPT = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)


def build_rag_chain(llm):
    return RAG_PROMPT | llm

## Agent 1: Router

Classifies the question into one of four types and selects the retrieval strategy.
- Conceptual questions → dense cosine retrieval (best for semantic similarity)
- Named-entity-heavy questions → BM25 (best for exact clinical terminology)
- Comparative and Multi-hop questions → hybrid RRF (maximises coverage across both arms)

**Multi-hop override:** If the golden dataset provides a `query_type` of `Multi-Hop` or `Multi-hop`,
the router skips the LLM call and directly assigns `Multi-hop` type with `hybrid` strategy.
This ensures multi-hop questions always get the broadest retrieval coverage.

In [8]:
ROUTER_PROMPT_TEMPLATE = """You are a biomedical question classifier.
Classify the given question into exactly one type and select the retrieval strategy.

Question types:
- Conceptual: asks about mechanisms, pathophysiology, or general relationships (e.g., "How does X cause Y?")
- Named-Entity: heavily involves specific drug names, gene variants, organisms, or identifiers (e.g., "Does BRCA1...")
- Comparative: compares two treatments, interventions, or conditions (e.g., "Does X outperform Y?")
- Multi-hop: requires combining evidence from multiple distinct studies or reasoning steps

Retrieval strategies:
- dense: cosine similarity embedding search (best for conceptual/semantic questions)
- bm25: keyword-based sparse retrieval (best for named entities and specific terminology)
- hybrid: combined dense + BM25 with rank fusion (best for comparative and multi-hop questions)

Respond with exactly this JSON format (no explanation, no markdown):
{{"question_type": "<Conceptual|Named-Entity|Comparative|Multi-hop>", "retrieval_strategy": "<dense|bm25|hybrid>"}}

Question: {question}

Classification:"""

ROUTER_PROMPT = PromptTemplate(
    template=ROUTER_PROMPT_TEMPLATE,
    input_variables=["question"],
)


def build_router_chain(llm):
    return ROUTER_PROMPT | llm


def route_question(question, query_type, router_chain):
    """Route a question to the appropriate retrieval strategy.

    If the golden dataset query_type is Multi-hop/Multi-Hop, bypass the LLM router
    and directly assign hybrid retrieval. For all other query_types, use the LLM router.

    Args:
        question: Question text.
        query_type: query_type value from the golden dataset (e.g. "Multi-Hop", "Single-hop").
        router_chain: LLM router chain for non-multi-hop questions.

    Returns:
        Tuple of (question_type str, retrieval_strategy str).
    """
    if str(query_type).strip().lower() == "multi-hop":
        return "Multi-hop", "hybrid"
    router_result = router_chain.invoke({"question": question})
    return parse_router_output(router_result)


def parse_router_output(raw_output):
    """Parse JSON router output, with fallback to dense retrieval on parse failure."""
    text = raw_output.content if hasattr(raw_output, "content") else str(raw_output)
    text = text.strip()
    # Strip markdown fences if present
    text = re.sub(r"```(?:json)?\s*", "", text).strip().rstrip("`")
    try:
        result = json.loads(text)
        question_type = result.get("question_type", "Conceptual")
        strategy = result.get("retrieval_strategy", "dense")
        if strategy not in ("dense", "bm25", "hybrid"):
            strategy = "dense"
        return question_type, strategy
    except json.JSONDecodeError:
        # Fallback: try to extract strategy from raw text
        for strat in ("hybrid", "bm25", "dense"):
            if strat in text.lower():
                return "Conceptual", strat
        return "Conceptual", "dense"

## Agent 3: Evidence Selection (Pass-Through)

The Evidence Grader originally re-ranked retrieved chunks using rule-based heuristics
(section label priority, study design keywords, MeSH term overlap). Empirical analysis
showed this heuristic **actively hurt** both faithfulness and correctness:

- The grader changed the top-5 selection in 174/200 questions (87%)
- Questions where the grader dropped a retriever top-3 chunk had mean faithfulness
  of 0.9513 vs 0.9838 for questions where it kept the top-3
- Two questions scored 0.0 faithfulness because the grader filtered out the only
  relevant chunks, causing the generator to refuse
- Head-to-head vs Hybrid+Cross-Encoder (which uses the same retriever without grading):
  essentially a coin flip with more catastrophic failures

The retriever's semantic ranking (dense cosine or cross-encoder) already captures
relevance better than keyword/section heuristics. The grader now simply passes through
the retriever's top-k chunks without re-ranking.

In [9]:
def select_top_chunks(docs, question, question_type, top_k=5):
    """Return the top-k chunks from the retriever's ranking (no re-ranking).

    The retriever's semantic ordering (dense cosine / cross-encoder / hybrid RRF)
    already captures relevance better than rule-based heuristics. This function
    simply truncates to top_k to maintain the same interface for downstream callers.
    """
    return docs[:top_k]

## Agent 5: Hallucination Guard (Detection-Only)

One LLM call after generation. The guard receives the generated answer and the retrieved
context chunks and identifies factual claims not supported by the context.

**Detection-only design:** The guard **never modifies the answer**. It returns the original
answer unchanged and only populates `hallucination_detected` and `unsupported_claims` for
downstream analysis. This prevents the guard from introducing refusal language or degrading
answer quality — a problem observed in earlier iterations where the guard rewrote answers
and caused faithfulness to drop.

**Single unified prompt:** Both high-risk and medium/low-risk questions use the same prompt.
The guard flags:
- Direct contradictions with the context (all questions)
- Specific clinical facts (drug names, dosages, statistics) with no basis in the context

The guard does NOT flag reasonable inferences, conclusions drawn from evidence, or
document ID references.

In [10]:
HALLUCINATION_GUARD_TEMPLATE = """You are a biomedical fact-checker reviewing a generated answer against its source context.

Context:
{context}

Generated Answer:
{answer}

Instructions:
1. Flag claims that DIRECTLY CONTRADICT the context.
2. Flag specific clinical facts (drug names, dosages, procedures, statistics) that appear in the answer but have NO basis in the context.
3. Do NOT flag reasonable inferences or conclusions drawn from the context.
4. Do NOT flag document ID references — ignore any mention of document identifiers.
5. ALWAYS return the original answer unchanged in verified_answer — do not rewrite or modify it.

Respond with exactly this JSON format (no explanation, no markdown):
{{"verified_answer": "<original answer unchanged>", "hallucination_detected": <true|false>, "unsupported_claims": ["<claim 1>", "<claim 2>"]}}

Response:"""

HALLUCINATION_GUARD_PROMPT = PromptTemplate(
    template=HALLUCINATION_GUARD_TEMPLATE,
    input_variables=["context", "answer"],
)


def build_hallucination_guard_chain(llm, severity_tier="Medium"):
    """Build the detection-only guard chain (single prompt for all severity tiers)."""
    return HALLUCINATION_GUARD_PROMPT | llm


def parse_guard_output(raw_output, fallback_answer):
    """Parse JSON hallucination guard output with graceful fallback.

    Returns:
        Tuple of (verified_answer str, hallucination_detected bool, unsupported_claims list).
    """
    text = raw_output.content if hasattr(raw_output, "content") else str(raw_output)
    text = text.strip()
    text = re.sub(r"```(?:json)?\s*", "", text).strip().rstrip("`")
    try:
        result = json.loads(text)
        detected = bool(result.get("hallucination_detected", False))
        claims = result.get("unsupported_claims", [])
        return fallback_answer, detected, claims
    except json.JSONDecodeError:
        return fallback_answer, False, []

## Evaluation Functions

In [11]:
class GroqModel(DeepEvalBaseLLM):
    def __init__(self, model=None):
        self.model = model or get_groq_llm()

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        response = self.model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "Groq Model"


def build_test_cases(eval_df):
    return [
        LLMTestCase(
            input=row["question"],
            actual_output=row["generated_answer"],
            retrieval_context=row["retrieved_contexts"],
            expected_output=row["golden_answer"],
        )
        for _, row in eval_df.iterrows()
    ]


def _make_ragas_scores_df(all_scores, metric_name):
    return pd.DataFrame({
        "question_index": range(len(all_scores)),
        metric_name: all_scores,
    })


def evaluate_ragas(eval_df, metric, results_file=None):
    """Evaluate with a non-LLM RAGAS metric over each row of the eval DataFrame."""
    metric_name = type(metric).__name__
    all_scores = []

    for _, row in eval_df.iterrows():
        sample = SingleTurnSample(
            user_input=row["question"],
            retrieved_contexts=list(row["retrieved_contexts"]),
            reference_contexts=row["golden_contexts"],
            reference=row["golden_answer"],
            response=row["generated_answer"]
        )
        score = metric.single_turn_score(sample)
        all_scores.append(score)

    avg = sum(all_scores) / len(all_scores)
    print(f"\n=== {metric_name}: {avg:.4f} (avg over {len(all_scores)} samples) ===")

    scores_df = _make_ragas_scores_df(all_scores, metric_name)
    if results_file:
        scores_df.to_csv(results_file, index=False)
        print(f"Saved scores to {results_file}")

    return all_scores, avg, scores_df


def build_ragas_combined(eval_df, score_dfs, results_file=None):
    combined = eval_df.copy().reset_index(drop=True)
    combined.insert(0, "question_index", range(len(combined)))
    combined = combined.rename(columns={"generated_answer": "generated_response"})
    for scores_df in score_dfs:
        combined = combined.merge(scores_df, on="question_index", how="left")
    if results_file:
        combined.to_csv(results_file, index=False)
        print(f"Saved combined RAGAS results to {results_file}")
    return combined


def _run_deepeval_slice(test_case_slice, api_key, model, metric_cls, threshold, delay, key_idx, metric_kwargs=None):
    llm = ChatGroq(model=model, api_key=api_key)
    results = []
    for i, test_case in enumerate(test_case_slice):
        try:
            metric = metric_cls(threshold=threshold, model=GroqModel(model=llm),
                                **metric_kwargs)
            result = deepeval.evaluate([test_case], metrics=[metric],
                                       display_config=DisplayConfig(
                                           verbose_mode=False,
                                           show_indicator=False,
                                           print_results=False),
                                       async_config=AsyncConfig(run_async=False))
            results.extend(result.test_results)
        except Exception as e:
            print(f"[Key {key_idx}] Error on case {i + 1}: {e}")
        if i < len(test_case_slice) - 1:
            time.sleep(delay)
    print(f"[Key {key_idx}] Done — {len(results)}/{len(test_case_slice)} cases evaluated")
    return results


def evaluate_deepeval_parallel(test_cases, metric_cls, key_rotator, threshold=0.5, results_file=None, delay=None, rows_per_key=None, metric_kwargs=None):
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.DEEPEVAL_DELAY_SECONDS
    api_keys = key_rotator.api_keys
    metric_kwargs = {} if metric_kwargs is None else dict(metric_kwargs)

    if not test_cases:
        raise ValueError("test_cases is empty.")

    total_capacity = len(api_keys) * rows_per_key
    if len(test_cases) > total_capacity:
        test_cases = test_cases[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(test_cases):
            break
        slices.append((key, i, test_cases[start: start + rows_per_key]))

    print(f"\n{len(test_cases)} cases split across {len(slices)} key(s):")
    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(_run_deepeval_slice, s, key, key_rotator.model,
                            metric_cls, threshold, delay, i, metric_kwargs): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                ordered_results[idx] = []

    all_results = []
    res_df = None
    for key_results in ordered_results:
        all_results.extend(key_results)

    if all_results:
        scores = [r.metrics_data[0].score for r in all_results]
        metric_name = all_results[0].metrics_data[0].name
        average = sum(scores) / len(scores)
        print(f"\n=== {metric_name}: {average:.4f} (avg over {len(scores)} samples) ===")
        rows = []
        for r in all_results:
            rows.append({
                "question": r.input,
                "generated_answer": r.actual_output,
                "retrieved_contexts": r.retrieval_context,
                "golden_answer": r.expected_output,
                r.metrics_data[0].name: r.metrics_data[0].score,
            })
        res_df = pd.DataFrame(rows)
        if results_file:
            res_df.to_csv(results_file, index=False)
            print(f"Saved to {results_file}")

    return all_results, res_df

## Evidence-Graded RAG Pipeline

The `_run_slice` function implements the full 5-agent pipeline per row:
1. **Agent 1 — Router:** Classifies question type and selects retrieval strategy (1 LLM call,
   or bypassed for multi-hop questions using golden dataset `query_type`).
2. **Agent 2 — Adaptive Retrieval:** Runs the selected retriever with k=8 to create a wider
   candidate pool.
3. **Agent 3 — Evidence Selection (pass-through):** Returns the retriever's top-5 chunks
   without re-ranking. The original heuristic grader was removed after analysis showed it
   degraded both faithfulness and correctness.
4. **Agent 4 — Answer Generator:** Generates an answer from the top-5 context chunks (1 LLM call).
   The prompt encourages synthesis and evidence-based inference rather than refusals.
5. **Agent 5 — Hallucination Guard (detection-only):** Checks for contradictions and unsupported
   clinical facts (1 LLM call). Flags issues in `hallucination_detected` and `unsupported_claims`
   but **never modifies** the generated answer.

In [12]:
CANDIDATE_K = 8   # Wider candidate pool for the evidence grader to work from
GRADED_TOP_K = 5  # Final number of chunks passed to the generator

SEVERITY_CACHE_PATH = config.DATA_PROCESSED_DIR / "golden_dataset_with_severity.csv"


def _load_severity_map():
    """Load the cached severity tier map from the clinical severity analysis.

    Returns:
        Dict mapping lowercase question text to severity tier string ("High", "Medium", "Low").
    """
    if not SEVERITY_CACHE_PATH.exists():
        print(f"Warning: severity cache not found at {SEVERITY_CACHE_PATH}. "
              "Run clinical_severity.ipynb first. Defaulting all to Medium.")
        return {}
    sev_df = pd.read_csv(SEVERITY_CACHE_PATH)
    return dict(zip(sev_df["question"].str.strip().str.lower(), sev_df["severity_tier"]))


def _run_slice(retrievers, slice_df, api_key, model, delay, key_idx, severity_map=None):
    """Process a contiguous slice of the evaluation set using the 5-agent pipeline.

    Args:
        retrievers: Dict with keys 'dense', 'bm25' mapping to retriever objects.
        slice_df: Contiguous DataFrame slice assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name.
        delay: Seconds to sleep between rows.
        key_idx: Key index used only for log prefixes.
        severity_map: Dict mapping lowercase question → severity tier.

    Returns:
        Copy of slice_df with new columns: route_type, route_strategy, retrieved_contexts,
        graded_contexts, generated_answer, hallucination_detected, unsupported_claims,
        and timing/token columns.
    """
    severity_map = severity_map or {}
    llm = ChatGroq(model=model, api_key=api_key)
    router_chain = build_router_chain(llm)
    rag_chain = build_rag_chain(llm)

    dense_retriever = retrievers["dense"]
    bm25_retriever = retrievers["bm25"]

    result_df = slice_df.copy().reset_index(drop=True)
    route_type_list = [None] * len(slice_df)
    route_strategy_list = [None] * len(slice_df)
    retrieved_contexts_list = [None] * len(slice_df)
    graded_contexts_list = [None] * len(slice_df)
    generated_answer_list = [None] * len(slice_df)
    hallucination_detected_list = [None] * len(slice_df)
    unsupported_claims_list = [None] * len(slice_df)
    total_time_list = [None] * len(slice_df)
    prompt_tokens_list = [None] * len(slice_df)
    completion_tokens_list = [None] * len(slice_df)
    total_tokens_list = [None] * len(slice_df)

    for row_idx, (_, row) in enumerate(slice_df.iterrows()):
        question = row["question"]
        query_type = row.get("query_type", "")
        try:
            time_start = time.perf_counter()

            # Agent 1: Router (multi-hop override from golden dataset)
            question_type, strategy = route_question(question, query_type, router_chain)

            # Agent 2: Adaptive Retrieval (k=8 candidate pool)
            if strategy == "bm25":
                bm25_retriever.k = CANDIDATE_K
                candidate_docs = bm25_retriever.invoke(question)
            elif strategy == "hybrid":
                candidate_docs = invoke_hybrid_retriever(
                    question, dense_retriever, bm25_retriever, top_k=CANDIDATE_K
                )
            else:  # dense
                candidate_docs = dense_retriever.invoke(question)

            # Agent 3: Evidence Selection (pass-through — retriever ranking preserved)
            graded_docs = select_top_chunks(candidate_docs, question, question_type, top_k=GRADED_TOP_K)

            # Agent 4: Answer Generator
            gen_result = rag_chain.invoke({"context": graded_docs, "question": question})
            draft_answer = gen_result.content

            # Token usage from generation call
            token_usage = gen_result.response_metadata.get("token_usage", {})

            # Agent 5: Hallucination Guard (detection-only — flags but does not modify answer)
            severity_tier = severity_map.get(question.strip().lower(), "Medium")
            guard_chain = build_hallucination_guard_chain(llm, severity_tier=severity_tier)
            context_text = "\n\n".join(doc.page_content for doc in graded_docs)
            guard_result = guard_chain.invoke({"context": context_text, "answer": draft_answer})
            _, hal_detected, bad_claims = parse_guard_output(guard_result, draft_answer)

            total_time = time.perf_counter() - time_start

            route_type_list[row_idx] = question_type
            route_strategy_list[row_idx] = strategy
            retrieved_contexts_list[row_idx] = [doc.page_content for doc in candidate_docs]
            graded_contexts_list[row_idx] = [doc.page_content for doc in graded_docs]
            generated_answer_list[row_idx] = draft_answer
            hallucination_detected_list[row_idx] = hal_detected
            unsupported_claims_list[row_idx] = bad_claims
            total_time_list[row_idx] = total_time
            prompt_tokens_list[row_idx] = token_usage.get("prompt_tokens", 0)
            completion_tokens_list[row_idx] = token_usage.get("completion_tokens", 0)
            total_tokens_list[row_idx] = token_usage.get("total_tokens", 0)

        except Exception as e:
            print(f"[Key {key_idx}] Error on '{question[:50]}...': {e}")

        if row_idx < len(slice_df) - 1:
            time.sleep(delay)

    result_df["route_type"] = route_type_list
    result_df["route_strategy"] = route_strategy_list
    result_df["retrieved_contexts"] = retrieved_contexts_list
    result_df["graded_contexts"] = graded_contexts_list
    result_df["generated_answer"] = generated_answer_list
    result_df["hallucination_detected"] = hallucination_detected_list
    result_df["unsupported_claims"] = unsupported_claims_list
    result_df["total_time"] = total_time_list
    result_df["prompt_tokens"] = prompt_tokens_list
    result_df["completion_tokens"] = completion_tokens_list
    result_df["total_tokens"] = total_tokens_list

    completed = sum(1 for x in generated_answer_list if x is not None)
    print(f"[Key {key_idx}] Done — {completed}/{len(slice_df)} rows collected")
    return result_df


def run_evidence_rag_parallel(retrievers, df, key_rotator, rows_per_key=None, delay=None):
    """Assign a contiguous slice of rows to each API key and run all slices in parallel.

    Args:
        retrievers: Dict with keys 'dense' and 'bm25' mapping to retriever objects.
        df: DataFrame with at least question, golden_answer, and query_type columns.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        rows_per_key: Max rows assigned to each key (default config.PARALLEL_BUCKET_SIZE).
        delay: Seconds between rows (default config.PARALLEL_DELAY_SECONDS).

    Returns:
        A copy of df with new columns added by _run_slice.
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.PARALLEL_DELAY_SECONDS
    api_keys = key_rotator.api_keys

    if len(df) == 0:
        raise ValueError("DataFrame is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(df) > total_capacity:
        print(f"Warning: {len(df)} rows exceed capacity ({total_capacity}). Truncating.")
        df = df.iloc[:total_capacity]

    severity_map = _load_severity_map()

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(df):
            break
        slices.append((key, i, df.iloc[start: start + rows_per_key]))

    print(f"\n{len(df)} rows split across {len(slices)} key(s) ({rows_per_key} rows/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: rows {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} rows)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_slice,
                retrievers, s, key, key_rotator.model, delay, i, severity_map,
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            _, _, s = slices[idx]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                fallback_df = s.copy().reset_index(drop=True)
                for col in ["route_type", "route_strategy", "retrieved_contexts",
                             "graded_contexts", "generated_answer",
                             "hallucination_detected", "unsupported_claims"]:
                    fallback_df[col] = [None] * len(s)
                ordered_results[idx] = fallback_df

    final_df = pd.concat(ordered_results, ignore_index=True)
    completed = final_df["generated_answer"].notna().sum()
    print(f"\nCompleted {completed}/{len(df)} questions total")
    print(f"\nAverage Time Per Query: {final_df['total_time'].mean():.4f}s")
    print(f"\nAverage Total Tokens Per Query: {final_df['total_tokens'].mean():.1f}")
    hal_count = final_df["hallucination_detected"].sum()
    print(f"\nHallucinations detected: {hal_count}/{len(df)} questions ({100*hal_count/len(df):.1f}%)")
    if "route_strategy" in final_df.columns:
        print(f"\nRouting distribution:\n{final_df['route_strategy'].value_counts().to_string()}")
    if "route_type" in final_df.columns:
        print(f"\nQuestion type distribution:\n{final_df['route_type'].value_counts().to_string()}")
    return final_df

---
## Setup

In [ ]:
embedding_key = config.DEFAULT_EMBEDDING
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[embedding_key])
key_rotator = GroqKeyRotator()
llm = key_rotator.get_llm()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Embedding: {config.EMBEDDING_MODELS[embedding_key]}")
print(f"LLM: {key_rotator.model}")
print(f"Run timestamp: {timestamp}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initialized GroqKeyRotator with 10 API key(s)
Embedding: sentence-transformers/all-MiniLM-L6-v2
LLM: llama-3.3-70b-versatile
Run timestamp: 20260516_164252


## Prepare Evaluation Sample

In [ ]:
golden_df = pd.read_csv(config.DATA_PROCESSED_DIR / "golden_dataset_complete.csv")
print(f"Loaded {len(golden_df)} samples from golden_dataset_complete.csv")
golden_df['golden_contexts'] = golden_df['golden_contexts'].apply(literal_eval)
golden_df.info()

Loaded 200 samples from golden_dataset_complete.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   question_idx     200 non-null    int64 
 1   question         200 non-null    object
 2   golden_answer    200 non-null    object
 3   golden_contexts  200 non-null    object
 4   query_type       200 non-null    object
 5   pubids_needed    200 non-null    object
dtypes: int64(1), object(5)
memory usage: 9.5+ KB


## Load Vector Store and Build Retrievers

In [ ]:
vector_store = load_chroma(embeddings, db_name=f"{embedding_key}_pubmed_chromadb")

# Dense retriever with k=8 (wider candidate pool for evidence grader)
dense_retriever = get_cosine_retriever(vector_store, k=CANDIDATE_K)
bm25_retriever = get_bm25_retriever(vector_store, k=CANDIDATE_K)

retrievers = {"dense": dense_retriever, "bm25": bm25_retriever}

Loading ChromaDB from /content/vectorstores/minilm_pubmed_chromadb


## Smoke Test: Single Question

In [ ]:
smoke_result = run_evidence_rag_parallel(retrievers, golden_df.head(1), key_rotator)
smoke_result


1 rows split across 1 key(s) (20 rows/key max):
  Key 0: rows 0–0 (1 rows)

[Key 0] Done — 1/1 rows collected

Completed 1/1 questions total

Average Time Per Query: 3.1537s

Average Total Tokens Per Query: 2173.0

Hallucinations detected: 0/1 questions (0.0%)

Routing distribution:
route_strategy
dense    1

Question type distribution:
route_type
Conceptual    1


,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed,route_type,route_strategy,retrieved_contexts,graded_contexts,generated_answer,hallucination_detected,unsupported_claims,total_time,prompt_tokens,completion_tokens,total_tokens
0,0,Is there a relationship between rheumatoid art...,Based on data derived from self-reported healt...,"[1,412 individuals attending the University of...",Single-hop,['10783841'],Conceptual,dense,[CONCLUSIONS: Based on data derived from self-...,[CONCLUSIONS: Based on data derived from self-...,"Based on the provided context, there appears t...",False,[],3.153709,1761,412,2173


## Run Evidence-Graded RAG on Full Evaluation Set

In [ ]:
eval_dataset = run_evidence_rag_parallel(retrievers, golden_df, key_rotator)
eval_dataset.to_csv(
    str(config.RESULTS_EVALSETS_DIR / f"evidence_graded_rag_{embedding_key}_chroma_{timestamp}.csv"),
    index=False
)
print(f"Generated {len(eval_dataset)} answers")


200 rows split across 10 key(s) (20 rows/key max):
  Key 0: rows 0–19 (20 rows)
  Key 1: rows 20–39 (20 rows)
  Key 2: rows 40–59 (20 rows)
  Key 3: rows 60–79 (20 rows)
  Key 4: rows 80–99 (20 rows)
  Key 5: rows 100–119 (20 rows)
  Key 6: rows 120–139 (20 rows)
  Key 7: rows 140–159 (20 rows)
  Key 8: rows 160–179 (20 rows)
  Key 9: rows 180–199 (20 rows)

[Key 9] Error on 'Do ganglionated plexi (GP) ablation during Maze IV...': Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kr6pmr2ff3jbhg3emgs9qds4` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 9896, Requested 2157. Please try again in 265ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
[Key 3] Error on 'Blunt trauma in intoxicated patients: is computed ...': Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b

In [ ]:
len(eval_dataset[eval_dataset["generated_answer"].isna()])

3

In [ ]:
error_dataset = eval_dataset[eval_dataset["generated_answer"].isna()]
print(f"Number of errored rows: {len(error_dataset)}")
subset_golden_df = golden_df[golden_df["question"].isin(error_dataset["question"])]
subset_eval_df = run_evidence_rag_parallel(retrievers, subset_golden_df, key_rotator, rows_per_key=1, delay=4)
subset_eval_df

Number of errored rows: 3

3 rows split across 3 key(s) (1 rows/key max):
  Key 0: rows 0–0 (1 rows)
  Key 1: rows 1–1 (1 rows)
  Key 2: rows 2–2 (1 rows)

[Key 0] Done — 1/1 rows collected
[Key 1] Done — 1/1 rows collected
[Key 2] Done — 1/1 rows collected

Completed 3/3 questions total

Average Time Per Query: 3.0413s

Average Total Tokens Per Query: 2228.7

Hallucinations detected: 1/3 questions (33.3%)

Routing distribution:
route_strategy
hybrid    2
dense     1

Question type distribution:
route_type
Comparative    2
Conceptual     1


,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed,route_type,route_strategy,retrieved_contexts,graded_contexts,generated_answer,hallucination_detected,unsupported_claims,total_time,prompt_tokens,completion_tokens,total_tokens
0,65,Blunt trauma in intoxicated patients: is compu...,The incidence of abdominal injury in intoxicat...,[Physical examination to detect abdominal inju...,Single-hop,['10798511'],Comparative,hybrid,[BACKGROUND: Physical examination to detect ab...,[BACKGROUND: Physical examination to detect ab...,"Based on the provided context, computed tomogr...",False,[],2.609323,1668,388,2056
1,128,Are home sampling kits for sexually transmitte...,The widespread acceptability of using HSKs for...,[There is an urgent need to increase opportuni...,Single-hop,['19103915'],Conceptual,dense,[OBJECTIVE: There is an urgent need to increas...,[OBJECTIVE: There is an urgent need to increas...,"Based on the provided context, it appears that...",True,[the researchers considered HSK as a viable op...,2.930399,1889,313,2202
2,184,Do ganglionated plexi (GP) ablation during Maz...,GP ablation and LAA occlusion do not affect si...,[GP ablation shows no significant difference i...,Comparative,"['25985014', '27131771']",Comparative,hybrid,[BACKGROUND: We investigated the role of surgi...,[BACKGROUND: We investigated the role of surgi...,"Based on the provided context, it appears that...",False,[],3.584187,1887,541,2428


In [ ]:
for idx, row in subset_eval_df.iterrows():
    eval_dataset.at[row["question_idx"], "generated_answer"] = row["generated_answer"]
    eval_dataset.at[row["question_idx"], "route_type"] = row["route_type"]
    eval_dataset.at[row["question_idx"], "route_strategy"] = row["route_strategy"]
    eval_dataset.at[row["question_idx"], "retrieved_contexts"] = row["retrieved_contexts"]
    eval_dataset.at[row["question_idx"], "graded_contexts"] = row["graded_contexts"]
    eval_dataset.at[row["question_idx"], "hallucination_detected"] = row["hallucination_detected"]
    eval_dataset.at[row["question_idx"], "unsupported_claims"] = row["unsupported_claims"]
    eval_dataset.at[row["question_idx"], "total_time"] = row["total_time"]
    eval_dataset.at[row["question_idx"], "prompt_tokens"] = row["prompt_tokens"]
    eval_dataset.at[row["question_idx"], "completion_tokens"] = row["completion_tokens"]
    eval_dataset.at[row["question_idx"], "total_tokens"] = row["total_tokens"]

len(eval_dataset[eval_dataset["generated_answer"].isna()])

0

In [ ]:
eval_dataset.to_csv(str(config.RESULTS_EVALSETS_DIR / f"evidence_graded_rag_{embedding_key}_chroma_{timestamp}.csv"))

In [ ]:
print(f"{eval_dataset['total_time'].apply('mean')}s per question")
print(f"{eval_dataset['total_tokens'].apply('mean')} total tokens per question")

14.106084529295005s per question
2156.08 total tokens per question


### RAGAS Evaluation

In [ ]:
ragas_cr_scores, ragas_cr_avg, ragas_cr_df = evaluate_ragas(
    eval_dataset, NonLLMContextRecall(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"evidence_graded_rag_{embedding_key}_context_recall_{timestamp}.csv")
)


=== NonLLMContextRecall: 0.2121 (avg over 200 samples) ===
Saved scores to /content/results/ragas/evidence_graded_rag_minilm_context_recall_20260516_164252.csv


In [ ]:
ragas_cp_scores, ragas_cp_avg, ragas_cp_df = evaluate_ragas(
    eval_dataset, NonLLMContextPrecisionWithReference(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"evidence_graded_rag_{embedding_key}_context_precision_{timestamp}.csv")
)


=== NonLLMContextPrecisionWithReference: 0.2955 (avg over 200 samples) ===
Saved scores to /content/results/ragas/evidence_graded_rag_minilm_context_precision_20260516_164252.csv


In [ ]:
ragas_bleu_scores, ragas_bleu_avg, ragas_bleu_df = evaluate_ragas(
    eval_dataset, BleuScore(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"evidence_graded_rag_{embedding_key}_bleu_{timestamp}.csv")
)


=== BleuScore: 0.1116 (avg over 200 samples) ===
Saved scores to /content/results/ragas/evidence_graded_rag_minilm_bleu_20260516_164252.csv


In [ ]:
ragas_rouge_scores, ragas_rouge_avg, ragas_rouge_df = evaluate_ragas(
    eval_dataset, RougeScore(rouge_type="rougeL", mode="fmeasure"),
    results_file=str(config.RESULTS_RAGAS_DIR / f"evidence_graded_rag_{embedding_key}_rouge_{timestamp}.csv")
)


=== RougeScore: 0.1690 (avg over 200 samples) ===
Saved scores to /content/results/ragas/evidence_graded_rag_minilm_rouge_20260516_164252.csv


In [ ]:
combined_ragas = build_ragas_combined(
    eval_dataset,
    [ragas_cr_df, ragas_cp_df, ragas_bleu_df, ragas_rouge_df],
    results_file=str(config.RESULTS_RAGAS_DIR / f"evidence_graded_rag_{embedding_key}_combined_{timestamp}.csv")
)

Saved combined RAGAS results to /content/results/ragas/evidence_graded_rag_minilm_combined_20260516_164252.csv


### DeepEval Evaluation

In [14]:
### TO DELETE ###
timestamp = "20260516_164252"
embedding_key = config.DEFAULT_EMBEDDING
eval_dataset = pd.read_csv(str(config.RESULTS_EVALSETS_DIR / f"evidence_graded_rag_{embedding_key}_chroma_{timestamp}.csv"))
eval_dataset['retrieved_contexts'] = eval_dataset['retrieved_contexts'].apply(literal_eval)
eval_dataset['graded_contexts'] = eval_dataset['graded_contexts'].apply(literal_eval)
eval_dataset['golden_contexts'] = eval_dataset['golden_contexts'].apply(literal_eval)
eval_dataset['unsupported_claims'] = eval_dataset['unsupported_claims'].apply(literal_eval)
eval_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              200 non-null    int64  
 1   question_idx            200 non-null    int64  
 2   question                200 non-null    object 
 3   golden_answer           200 non-null    object 
 4   golden_contexts         200 non-null    object 
 5   query_type              200 non-null    object 
 6   pubids_needed           200 non-null    object 
 7   route_type              200 non-null    object 
 8   route_strategy          200 non-null    object 
 9   retrieved_contexts      200 non-null    object 
 10  graded_contexts         200 non-null    object 
 11  generated_answer        200 non-null    object 
 12  hallucination_detected  200 non-null    bool   
 13  unsupported_claims      200 non-null    object 
 14  total_time              200 non-null    fl

In [15]:
test_cases = build_test_cases(eval_dataset)
de_key_rotator = GroqKeyRotator(model="openai/gpt-oss-120b")

Initialized GroqKeyRotator with 10 API key(s)


In [ ]:
deepeval_cr, deepeval_cr_df = evaluate_deepeval_parallel(
    test_cases, ContextualRecallMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_ctx_recall_{timestamp}.csv")
)

In [ ]:
deepeval_cp, deepeval_cp_df = evaluate_deepeval_parallel(
    test_cases, ContextualPrecisionMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_ctx_precision_{timestamp}.csv")
)

In [ ]:
deepeval_f, deepeval_f_df = evaluate_deepeval_parallel(
    test_cases, FaithfulnessMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_faithfulness_{timestamp}.csv")
)


200 cases split across 10 key(s):


[Key 8] Done — 17/20 cases evaluated

=== Faithfulness: 0.9837 (avg over 189 samples) ===
Saved to /content/results/deepeval/evidence_graded_rag_minilm_faithfulness_20260516_164252.csv


In [18]:
def resume_deepeval_from_csv(eval_dataset, existing_results_file, metric_cls, key_rotator,
    metric_column, threshold=0.5, delay=None, rows_per_key=None, metric_kwargs=None):
    """Resume an interrupted DeepEval run from an existing CSV.

    Identifies missing or incomplete questions in an existing DeepEval results
    file, recomputes only those rows, merges the new scores, and overwrites the original file.

    Matching is performed using the question text rather than row position,
    making the method robust to out-of-order, partially completed, or shuffled CSV files.

    Args:
        eval_dataset: Full evaluation DataFrame.
        existing_results_file: Existing DeepEval CSV path.
        metric_cls: DeepEval metric class.
        key_rotator: API key rotator.
        metric_column: Metric column name in CSV.
        threshold: DeepEval threshold.
        delay: Delay between API calls.
        rows_per_key: Rows per API key.
        metric_kwargs: Optional metric init kwargs.

    Returns:
        Final merged DataFrame.
    """
    existing_df = pd.read_csv(existing_results_file)

    # Questions already successfully evaluated
    completed_questions = set(existing_df.loc[
        existing_df[metric_column].notna(),"question"].astype(str))

    # Missing/incomplete rows anywhere in dataset
    missing_df = eval_dataset[~eval_dataset["question"].astype(str)
                              .isin(completed_questions)].copy()
    if missing_df.empty:
        print("All rows already completed.")
        return existing_df
    print(f"Need to recompute {len(missing_df)} rows.")

    # Build test cases only for missing rows
    test_cases = build_test_cases(missing_df)

    # Run DeepEval only for missing rows
    _, new_results_df = evaluate_deepeval_parallel(test_cases, metric_cls,
        key_rotator, threshold=threshold, delay=delay, rows_per_key=rows_per_key,
        metric_kwargs=metric_kwargs)

    # Remove old incomplete duplicates
    existing_df = existing_df[~existing_df["question"].astype(str).isin(
            new_results_df["question"].astype(str))]

    # Merge
    final_df = pd.concat([existing_df, new_results_df], ignore_index=True)
    if "question_idx" in final_df.columns:
      final_df = final_df.drop(columns=["question_idx"])
    final_df = final_df.merge(eval_dataset[["question", "question_idx"]], on="question", how="left")
    cols = ["question_idx"] + [c for c in final_df.columns if c != "question_idx"]
    final_df = final_df.sort_values("question_idx").reset_index(drop=True)

    final_df.to_csv(existing_results_file, index=False)
    print(f"Completed: {len(final_df)}/{len(eval_dataset)} rows")

    return final_df

In [ ]:
faithfulness_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_faithfulness_{timestamp}.csv"), FaithfulnessMetric,
    de_key_rotator, "Faithfulness", delay=40, rows_per_key=4
)

Need to recompute 11 rows.

11 cases split across 3 key(s):


[Key 1] Done — 4/4 cases evaluated

=== Faithfulness: 0.9709 (avg over 10 samples) ===
Completed: 199/200 rows


In [ ]:
de_key_rotator = GroqKeyRotator(model="openai/gpt-oss-120b")
faithfulness_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_faithfulness_{timestamp}.csv"), FaithfulnessMetric,
    de_key_rotator, "Faithfulness", delay=40, rows_per_key=4
)

Initialized GroqKeyRotator with 1 API key(s)
Need to recompute 1 rows.

1 cases split across 1 key(s):


[Key 0] Done — 1/1 cases evaluated

=== Faithfulness: 1.0000 (avg over 1 samples) ===
Completed: 200/200 rows


In [ ]:
faithfulness_df['Faithfulness'].apply('mean')

np.float64(0.9831700382950381)

In [ ]:
# Defining Answer Correctness
evaluation_steps = [
    "Compare the generated answer with the reference answer in the context of the original biomedical question.",
    "Check whether the generated answer contains factually correct biomedical information and no contradictions to the reference answer.",
    "Verify that all clinically important facts needed to answer the question are present and no critical information is missing.",
    "Ignore wording differences, but penalize incorrect medical claims, unsupported conclusions, or misleading clinical interpretations."
]
deepeval_ac = evaluate_deepeval_parallel(
    test_cases, GEval, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_answer_correctness_{timestamp}.csv"),
    metric_kwargs={
        "name": "AnswerCorrectness",
        "evaluation_steps": evaluation_steps,
        "evaluation_params": [
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ]}
)


200 cases split across 10 key(s):
[Key 6] Error on case 1: Error code: 400 - {'error': {'message': 'Organization has been restricted. Please reach out to support if you believe this was in error.', 'type': 'invalid_request_error', 'code': 'organization_restricted'}}
[Key 5] Error on case 1: Error code: 400 - {'error': {'message': 'Organization has been restricted. Please reach out to support if you believe this was in error.', 'type': 'invalid_request_error', 'code': 'organization_restricted'}}
[Key 8] Error on case 1: Error code: 400 - {'error': {'message': 'Organization has been restricted. Please reach out to support if you believe this was in error.', 'type': 'invalid_request_error', 'code': 'organization_restricted'}}
[Key 7] Error on case 1: Error code: 400 - {'error': {'message': 'Organization has been restricted. Please reach out to support if you believe this was in error.', 'type': 'invalid_request_error', 'code': 'organization_restricted'}}


[Key 9] Done — 20/20 cases evaluated

=== AnswerCorrectness [GEval]: 0.8517 (avg over 120 samples) ===
Saved to /content/results/deepeval/evidence_graded_rag_minilm_answer_correctness_20260516_164252.csv


In [ ]:
de_key_rotator = GroqKeyRotator(model="openai/gpt-oss-120b")

answer_correctness_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_answer_correctness_{timestamp}.csv"),
    GEval, de_key_rotator, "AnswerCorrectness [GEval]", delay=40, rows_per_key=8,
    metric_kwargs={
        "name": "AnswerCorrectness",
        "evaluation_steps": evaluation_steps,
        "evaluation_params": [
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ]}
)

Initialized GroqKeyRotator with 10 API key(s)
Need to recompute 80 rows.

80 cases split across 10 key(s):
[Key 9] Error on case 1: 'NoneType' object has no attribute 'load'


[Key 0] Done — 8/8 cases evaluated

=== AnswerCorrectness [GEval]: 0.7859 (avg over 78 samples) ===
Completed: 198/200 rows


In [ ]:
answer_correctness_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_answer_correctness_{timestamp}.csv"),
    GEval, de_key_rotator, "AnswerCorrectness [GEval]", delay=40, rows_per_key=8,
    metric_kwargs={
        "name": "AnswerCorrectness",
        "evaluation_steps": evaluation_steps,
        "evaluation_params": [
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ]}
)

Need to recompute 2 rows.

2 cases split across 1 key(s):


[Key 0] Done — 2/2 cases evaluated

=== AnswerCorrectness [GEval]: 1.0000 (avg over 2 samples) ===
Completed: 200/200 rows


In [ ]:
answer_correctness_df['AnswerCorrectness [GEval]'].apply('mean')

np.float64(0.8275)

In [16]:
deepeval_ar, de_ar_df = evaluate_deepeval_parallel(
    test_cases, AnswerRelevancyMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_ans_relevancy_{timestamp}.csv")
)


200 cases split across 10 key(s):


[Key 7] Done — 20/20 cases evaluated

=== Answer Relevancy: 0.9336 (avg over 194 samples) ===
Saved to /content/results/deepeval/evidence_graded_rag_minilm_ans_relevancy_20260516_164252.csv


In [19]:
ans_relevancy_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_ans_relevancy_{timestamp}.csv"),
    AnswerRelevancyMetric, de_key_rotator, "Answer Relevancy", delay=40, rows_per_key=1
)

Need to recompute 6 rows.

6 cases split across 6 key(s):
[Key 3] Error on case 1: 'NoneType' object has no attribute 'save'
[Key 3] Done — 0/1 cases evaluated


[Key 0] Done — 1/1 cases evaluated

=== Answer Relevancy: 0.8733 (avg over 5 samples) ===
Completed: 199/200 rows


In [20]:
ans_relevancy_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_ans_relevancy_{timestamp}.csv"),
    AnswerRelevancyMetric, de_key_rotator, "Answer Relevancy", delay=40, rows_per_key=1
)

Need to recompute 1 rows.

1 cases split across 1 key(s):


[Key 0] Done — 1/1 cases evaluated

=== Answer Relevancy: 0.7778 (avg over 1 samples) ===
Completed: 200/200 rows


In [21]:
ans_relevancy_df['Answer Relevancy'].apply('mean')

np.float64(0.9313215738509857)

In [ ]:
import shutil
shutil.copytree('/content/results', '/content/drive/MyDrive/LJMU/results')

'/content/drive/MyDrive/LJMU/results'